# 示例一：使用 OpenRouter 提取关键词

学习顺序：读取配置 → 创建客户端 → 定义函数 → 调用示例。

本文件独立运行，内核选择 `Python (course1_365)`。另一个示例见 [课程网页 RAG](ChatGPT_OpenRouter_RAG.ipynb)。两个文件共用同目录 `.env`，无需复制 Key。


## 1. 读取配置

`load_dotenv` 读取同目录的 `.env`；`os.getenv` 取出配置值。修改 `.env` 后重新运行此单元格即可。


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Notebook 通常以所在文件夹为工作目录；若从别处启动，请修改 BASE_DIR。
# 不在代码中写 Key，也不打印 Key，便于分享 Notebook。
BASE_DIR = Path.cwd()
load_dotenv(BASE_DIR / ".env", override=True)

# strip() 去掉配置前后的空格；提前检查，比请求时才报错更容易定位。
api_key = os.getenv("OPENROUTER_API_KEY", "").strip()
model = os.getenv("OPENROUTER_MODEL", "").strip()
if not api_key or not model:
    raise ValueError("请在同目录 .env 配置 OPENROUTER_API_KEY 和 OPENROUTER_MODEL。")


## 2. 创建客户端

`api_key` 使用 OpenRouter 的 Key，`base_url` 指定请求发送到 OpenRouter。


In [ ]:
# OpenRouter 提供兼容 OpenAI 的接口，因此这里使用 OpenAI Python SDK。
# 创建客户端不会发送聊天请求；真正的请求发生在函数调用时。
client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    timeout=60,     # 单次网络请求的超时时间（秒）。
    max_retries=2,  # 暂时性网络/服务错误最多重试两次。
)


## 3. 定义关键词提取函数

在 `messages` 中放入两组 `user` / `assistant` 示例，让模型参考提取方式和输出格式，这叫少样本提示（few-shot prompting）。这些是我们提供的示范答案。

最后一条 `user` 才是本次待处理的文本。每次调用都使用新的消息列表，各次测试互不保留聊天历史。不设置输出 token 上限。


In [ ]:
def extract_keywords(text: str) -> str:
    """提取文本关键词，返回中文逗号分隔的字符串。

    每次调用独立：示范消息用于说明输出格式，不会记住上次输入。
    这里不设置输出 token 上限，但服务端仍受模型自身限制。
    """
    if not isinstance(text, str) or not text.strip():
        raise ValueError("请输入非空文本。")

    # system：任务规则；user/assistant 示例：展示我们期望的输入和答案。
    # 最后一条 user 消息才是本次真正需要处理的文本。
    messages = [
        {"role": "system", "content":
         "请提取用户文本中的核心关键词或短语，去除重复，只返回关键词，用中文逗号分隔。"},
        {"role": "user", "content": "我正在学习 Python，希望用 pandas 分析销售数据。"},
        {"role": "assistant", "content": "Python，pandas，销售数据分析"},
        {"role": "user", "content": "这家酒店房间干净，交通方便，但晚上噪音比较大。"},
        {"role": "assistant", "content": "酒店，房间干净，交通方便，夜间噪音"},
        {"role": "user", "content": text.strip()},
    ]
    response = client.chat.completions.create(model=model, messages=messages)

    # choices 是候选答案列表；通常取第一个候选的文字内容。
    answer = response.choices[0].message.content
    if not answer or not answer.strip():
        raise ValueError("模型没有返回文字，请检查所选模型或稍后重试。")
    return answer.strip()


## 4. 调用函数

每次调用会发送一次 API 请求。后续新增代码单元格，调用 `extract_keywords("你的文本")` 即可。


In [ ]:
# 修改 text 即可复用函数；执行此单元格会调用 OpenRouter。
text = "大语言模型可以分析客户反馈、提取关键词，并生成结构化报告。"
print(extract_keywords(text))


也可以手动输入一段文本：

In [ ]:
# 可选的交互示例：运行后输入文本并按 Enter。
text = input("请输入需要提取关键词的文本：")
print(extract_keywords(text))
